In [1]:
import os
import pandas as pd

base_path = "../data/answers/answers"

folders = [
    "r4.2-1",
    "r4.2-2",
    "r4.2-3"
]

files = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            files.append(os.path.join(folder_path, file))

print("Total r4.2 scenario files:", len(files))

Total r4.2 scenario files: 70


In [2]:
def parse_r42_file(file_path):
    events = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split(",")
            event_type = parts[0]

            if event_type in ["logon", "device"]:
                if len(parts) >= 6:
                    events.append({
                        "event_type": parts[0],
                        "event_id": parts[1],
                        "timestamp": parts[2],
                        "user": parts[3],
                        "pc": parts[4],
                        "action": parts[5],
                        "url": "",
                        "keywords": ""
                    })

            elif event_type == "http":
                if len(parts) >= 7:
                    events.append({
                        "event_type": parts[0],
                        "event_id": parts[1],
                        "timestamp": parts[2],
                        "user": parts[3],
                        "pc": parts[4],
                        "action": "",
                        "url": parts[5],
                        "keywords": ",".join(parts[6:])
                    })

    return pd.DataFrame(events)

In [3]:
all_events = []

for file in files:

    df_temp = parse_r42_file(file)

    df_temp["scenario"] = os.path.basename(file).replace(".csv", "")

    all_events.append(df_temp)

r42_all_events = pd.concat(
    all_events,
    ignore_index=True
)

print("Total events:", len(r42_all_events))
print("Total scenarios:", r42_all_events["scenario"].nunique())
print("Total users:", r42_all_events["user"].nunique())

Total events: 6843
Total scenarios: 70
Total users: 72


In [4]:
r42_all_events["timestamp"] = pd.to_datetime(
    r42_all_events["timestamp"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

print(r42_all_events.dtypes)

event_type               str
event_id                 str
timestamp     datetime64[us]
user                     str
pc                       str
action                   str
url                      str
keywords                 str
scenario                 str
dtype: object


In [5]:
r42_all_events.to_csv(
    "../data/processed/r42_all_parsed_events.csv",
    index=False
)

In [6]:
df = r42_all_events.copy()

df["hour"] = df["timestamp"].dt.hour

df["after_hours"] = (
    (df["hour"] < 7) |
    (df["hour"] >= 19)
).astype(int)

df["is_http"] = (
    df["event_type"] == "http"
).astype(int)

df["is_wikileaks"] = (
    df["url"]
    .fillna("")
    .str.lower()
    .str.contains("wikileaks", na=False)
).astype(int)

df["is_connect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Connect")
).astype(int)

df["is_disconnect"] = (
    (df["event_type"] == "device") &
    (df["action"] == "Disconnect")
).astype(int)

df["is_logon"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logon")
).astype(int)

df["is_logoff"] = (
    (df["event_type"] == "logon") &
    (df["action"] == "Logoff")
).astype(int)

df["after_hours_logon_event"] = (
    (df["is_logon"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

df["after_hours_http_event"] = (
    (df["is_http"] == 1) &
    (df["after_hours"] == 1)
).astype(int)

In [7]:
scenario_features_all = df.groupby(
    ["scenario", "user"]
).agg(
    scenario_start=("timestamp", "min"),
    scenario_end=("timestamp", "max"),
    total_events=("event_type", "count"),
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    device_connect_count=("is_connect", "sum"),
    device_disconnect_count=("is_disconnect", "sum"),
    http_count=("is_http", "sum"),
    after_hours_events=("after_hours", "sum"),
    after_hours_logon=("after_hours_logon_event", "sum"),
    after_hours_http=("after_hours_http_event", "sum"),
    wikileaks_access=("is_wikileaks", "sum"),
    unique_pcs=("pc", "nunique")
).reset_index()

scenario_features_all["scenario_duration_hours"] = (
    scenario_features_all["scenario_end"] -
    scenario_features_all["scenario_start"]
).dt.total_seconds() / 3600

scenario_features_all["insider_label"] = 1

print("Shape:", scenario_features_all.shape)

Shape: (80, 17)


In [8]:
scenario_features_all.to_csv(
    "../data/processed/r42_all_scenario_features.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [9]:
for folder in folders:
    folder_path = os.path.join(base_path, folder)

    count = len([
        f for f in os.listdir(folder_path)
        if f.endswith(".csv")
    ])

    print(folder, ":", count)

r4.2-1 : 30
r4.2-2 : 30
r4.2-3 : 10


In [10]:
print(
    scenario_features_all
    .groupby(scenario_features_all["scenario"].str[:6])
    .size()
)

scenario
r4.2-1    30
r4.2-2    30
r4.2-3    20
dtype: int64


In [11]:
print(
    scenario_features_all["scenario"].head(10).tolist()
)

['r4.2-1-AAM0658', 'r4.2-1-AJR0932', 'r4.2-1-BDV0168', 'r4.2-1-BIH0745', 'r4.2-1-BLS0678', 'r4.2-1-BTL0226', 'r4.2-1-CAH0936', 'r4.2-1-DCH0843', 'r4.2-1-EHB0824', 'r4.2-1-EHD0584']


In [12]:
insiders_file = "../data/answers/answers/insiders.csv"

insiders = pd.read_csv(
    insiders_file,
    header=None
)

r42_insiders = insiders[
    insiders[0].astype(str).str.strip() == "4.2"
]

print("r4.2 insider records:", len(r42_insiders))
print(r42_insiders.head())

r4.2 insider records: 70
      0  1                   2        3                    4  \
9   4.2  1  r4.2-1-AAM0658.csv  AAM0658  10/23/2010 01:34:19   
10  4.2  1  r4.2-1-AJR0932.csv  AJR0932  09/10/2010 19:12:01   
11  4.2  1  r4.2-1-BDV0168.csv  BDV0168  07/30/2010 19:56:44   
12  4.2  1  r4.2-1-BIH0745.csv  BIH0745  07/13/2010 20:15:23   
13  4.2  1  r4.2-1-BLS0678.csv  BLS0678  09/21/2010 01:16:22   

                      5  
9   10/29/2010 05:23:28  
10  09/18/2010 02:02:51  
11  08/10/2010 05:16:41  
12  07/13/2010 21:20:44  
13  09/30/2010 04:48:19  


In [13]:
print("Our parsed scenarios:", scenario_features_all["scenario"].nunique())

print("\nScenario counts by folder:")
print(
    scenario_features_all["scenario"]
    .str.extract(r"^(r4\.2-\d)")
    [0]
    .value_counts()
    .sort_index()
)

Our parsed scenarios: 70

Scenario counts by folder:
0
r4.2-1    30
r4.2-2    30
r4.2-3    20
Name: count, dtype: int64


In [14]:
# Get officially labelled r4.2 insider scenario names
official_insider_scenarios = set(
    r42_insiders[2]
    .astype(str)
    .str.replace(".csv", "", regex=False)
)

# Get all parsed scenario names
all_scenarios = set(
    scenario_features_all["scenario"]
)

# Find scenarios that are NOT officially labelled as insiders
normal_scenarios = sorted(
    all_scenarios - official_insider_scenarios
)

print("Total parsed scenarios:", len(all_scenarios))
print("Official insider scenarios:", len(official_insider_scenarios))
print("Potential normal scenarios:", len(normal_scenarios))

print("\nPotential normal scenarios:")
print(normal_scenarios)

Total parsed scenarios: 70
Official insider scenarios: 70
Potential normal scenarios: 0

Potential normal scenarios:
[]


In [15]:
r1_features = pd.read_csv(
    "../data/processed/leakmind_features.csv"
)

print("Shape:", r1_features.shape)
print(r1_features.head())

Shape: (1000, 13)
           user  http_activity_count  after_hours_activity  \
0  DTAA/AAA0371                 7085                     0   
1  DTAA/AAC0344                14884                     0   
2  DTAA/AAC0599                14926                     0   
3  DTAA/AAH0734                16740                     0   
4  DTAA/AAK0658                14715                     0   

   total_logon_events  unique_pcs  logon_count  logoff_count  \
0                 843          16          536           307   
1                 690           1          345           345   
2                 690           1          345           345   
3                 690           1          345           345   
4                 690           1          345           345   

   after_hours_logon  total_device_events  device_pcs  device_connect_count  \
0                0.0                  0.0         0.0                   0.0   
1                0.0                  0.0         0.0             

In [16]:
print(r1_features.columns.tolist())

['user', 'http_activity_count', 'after_hours_activity', 'total_logon_events', 'unique_pcs', 'logon_count', 'logoff_count', 'after_hours_logon', 'total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device']


In [17]:
print(
    r1_features[
        [
            "http_activity_count",
            "after_hours_activity",
            "total_logon_events",
            "unique_pcs",
            "after_hours_logon",
            "total_device_events"
        ]
    ].describe()
)

       http_activity_count  after_hours_activity  total_logon_events  \
count          1000.000000           1000.000000         1000.000000   
mean           3451.665000             64.683000          849.579000   
std            3868.476028            205.674034          496.002531   
min               5.000000              0.000000            8.000000   
25%            1637.000000              0.000000          690.000000   
50%            1935.500000              0.000000          690.000000   
75%            2527.000000             20.250000          973.000000   
max           20966.000000           2533.000000         5110.000000   

        unique_pcs  after_hours_logon  total_device_events  
count  1000.000000        1000.000000          1000.000000  
mean     22.467000          35.026000            65.668000  
std      96.635528         132.488908           153.361088  
min       1.000000           0.000000             0.000000  
25%       1.000000           0.000000         